### 데이터 분할
1. 기본 아이디어
   - 시간 기준 분할 : Train(2010-2019) / Test(2020-2025)
     - why? 과거 제소 이력으로 학습 -> 현재 데이터 보고 -> 미래 제소 예측
       따라서 랜덤 분할 시 미래 보고 과거 예측하는 문제 발생
     - 2020년이 Train 25,143 / Test 23,949 → 가장 균등
2. 문제 존재
   - Train: 한국피소(1) 75%  비피소(0) 25%  ← 한국피소가 3배 많음
   - Test:  한국피소(1) 30%  비피소(0) 70%  ← 비피소가 2배 많음
3. 해결책(근데 해결이 될 지는 모르겠음)
   - class_weight='balanced' 적용

### 0. 설명
- 본인 모델 시작 전에 아래 두 코드 넣고 시작하면 됨.
- 우리 서비스는 **조기경보**이기 때문에, Recall을 최우선으로 높이는 방향으로 가야함.
- 그러나 나머지 지표가 쓰레기면 그것도 문제. 아래는 클로드가 정해준 지표 간 균형 기준임
  - Recall    ≥ 0.70  ← 최우선, 이것부터 맞추기
  - Precision ≥ 0.50  ← 최소 기준, 이 아래면 의미없음
  - AUC-ROC   ≥ 0.70  ← 전반적 성능 확인용
  - F1-score  ≥ 0.60  ← Recall·Precision 균형 확인용
---
<우리가 다룰 모델> - 한 명이 하나씩 잡으면 됨
1. Logistic Regression
2. Random Forest
3. XGBoost/LGBM

### 1. 공통 Train/Test 분할 코드
- ML 학습 수행 전에 이걸로 쓰면 됩니다.

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

df = pd.read_csv('ml_ready.csv', encoding='utf-8-sig')
df['start_ym'] = pd.PeriodIndex(df['start_ym'].astype(str), freq='M')

feature_cols = [c for c in df.columns
                if c not in ['is_korea_target', 'start_ym']]

train = df[df['start_ym'].dt.year < 2020]
test  = df[df['start_ym'].dt.year >= 2020]

X_train = train[feature_cols]
y_train = train['is_korea_target']
X_test  = test[feature_cols]
y_test  = test['is_korea_target']

### 2. 공통 평가 함수
- 일단 예측이니까 Recall이 젤 중요하긴 함

In [7]:
def evaluate(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n{'='*50}")
    print(f"모델: {name}")
    print(f"{'='*50}")
    print(classification_report(
        y_test, y_pred,
        target_names=['비피소(0)', '한국피소(1)']
    ))
    print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.3f}")